In [ ]:
!apt-get update
!apt-get install -y cd-hit

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,970 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.0 MB]
Get:14 http://security.ub

In [ ]:
!pip install biopython scikit-learn numpy pandas transformers torch -q

In [ ]:
!cd-hit -i signal_peptides.fasta -o signal_filtered.fasta -c 0.8 -n 5
!cd-hit -i nonsignal_peptides.fasta -o nonsignal_filtered.fasta -c 0.8 -n 5

Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Command: cd-hit -i signal_peptides.fasta -o
         signal_filtered.fasta -c 0.8 -n 5

Started: Mon Apr 20 04:21:28 2026
                            Output                              
----------------------------------------------------------------
total seq: 500
longest and shortest : 8384 and 64
Total letters: 341681
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 0M
Buffer          : 1 X 12M = 12M
Table           : 1 X 65M = 65M
Miscellaneous   : 0M
Total           : 78M

Table limit with the given memory limit:
Max number of representatives: 677007
Max number of word counting entries: 90243787

comparing sequences from          0  to        500

      500  finished        498  clusters

Approximated maximum memory consumption: 80M
writing new database
writing clustering information
program completed !

Total CPU time 0.34
Program: CD-HIT, V4.8.1 (+OpenMP), Aug 20 2021, 08:39:56
Comman

In [ ]:
import re
import torch
import numpy as np
import pandas as pd
from Bio import SeqIO
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
def parse_fasta(filepath):
    headers, seqs = [], []
    for record in SeqIO.parse(filepath, "fasta"):
        headers.append(record.id)
        seqs.append(str(record.seq))
    return headers, seqs

signal_headers, signal_seqs = parse_fasta("signal_filtered.fasta")
nonsignal_headers, nonsignal_seqs = parse_fasta("nonsignal_filtered.fasta")

all_headers = signal_headers + nonsignal_headers
all_seqs    = signal_seqs + nonsignal_seqs
labels      = [1] * len(signal_seqs) + [0] * len(nonsignal_seqs)

print(f"Signal: {len(signal_seqs)}, Non-signal: {len(nonsignal_seqs)}, Total: {len(all_seqs)}")

Signal: 498, Non-signal: 474, Total: 972


In [ ]:
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"

def get_physicochemical(seq):
    clean = ''.join([aa for aa in seq if aa in AMINO_ACIDS])
    if len(clean) < 5:
        return np.zeros(6)
    try:
        a = ProteinAnalysis(clean)
        return np.array([
            a.gravy(),                           # hydrophobicity
            a.isoelectric_point(),               # isoelectric point
            a.molecular_weight(),                # molecular weight
            a.instability_index(),               # instability index
            a.aromaticity(),                     # aromaticity
            a.secondary_structure_fraction()[0]  # helix propensity
        ])
    except:
        return np.zeros(6)

physchem = np.array([get_physicochemical(s) for s in all_seqs])
physchem_scaled = StandardScaler().fit_transform(physchem)
print("Physicochemical shape:", physchem_scaled.shape)  # (n, 6)

Physicochemical shape: (972, 6)


In [ ]:
print("Loading BioBERT...")
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")
model = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")
model.eval()

bert_embeddings = []

for i, seq in enumerate(all_seqs):
    spaced_seq = ' '.join(seq[:500])
    inputs = tokenizer(spaced_seq, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        emb = outputs.last_hidden_state[:, 0, :].numpy()
        bert_embeddings.append(emb[0])
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(all_seqs)} done")

bert_embeddings = np.array(bert_embeddings)
print("BioBERT shape:", bert_embeddings.shape)  # (n, 768)

Loading BioBERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  100/972 done
  200/972 done
  300/972 done
  400/972 done
  500/972 done
  600/972 done
  700/972 done
  800/972 done
  900/972 done
BioBERT shape: (972, 768)


In [ ]:
# combine BERT (768) + physicochemical (6) = 774 features
X = np.hstack([bert_embeddings, physchem_scaled])
y = np.array(labels)

print(f"Final X shape: {X.shape}")  # (n, 774)

# save as npy
np.save("X_features.npy", X)
np.save("y_labels.npy", y)

# save as CSV
df = pd.DataFrame(bert_embeddings, columns=[f'emb_{i}' for i in range(768)])
df['gravy']       = physchem[:, 0]
df['isoelectric'] = physchem[:, 1]
df['mol_weight']  = physchem[:, 2]
df['instability'] = physchem[:, 3]
df['aromaticity'] = physchem[:, 4]
df['helix']       = physchem[:, 5]
df['label']       = labels
df['id']          = all_headers
df.to_csv("features_all.csv", index=False)

print(f"Saved {len(all_seqs)} sequences, {X.shape[1]} features each")

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")